## **Context**




We are going to use a large language model to automate the classification and processing of user help desk support tickets.  The ultimate goal would be to predict ticket categories, assign priority, suggest estimated resolution time, generate a response based on sentiment analysis from the LLM, and create output that is stored in a dataframe. The input file is Support_ticket_text_data.xls.

The dateframe should have 7 columns:

Support ticket ID (from input file), support ticket text (from input file), category, tags, priority, estimated resolution time, and a generated reply from the LLM.

You will likely need to run this from Google Colab.

## **Project Objective**

Develop a Generative AI application using a Large Language Model to **automate the classification and processing of support tickets.** The application will aim to predict ticket categories, assign priority, suggest estimated resolution times, generate responses based on sentiment analysis, and store the results in a structured DataFrame.


## **Model Loading**

In [1]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --upgrade --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 210.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 202.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 275.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.2.1+cu121 requires nvidia-cublas-cu12==12.1.3.1; platform_system == "Linux" and platform_machine == "x86_64", which is not installed.
torch 2.2.1+cu121 requires nvidia-cuda-cupti-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86_64", which is not installed.
torch 2.2.1+cu121 requires nvi

In [2]:
# Install the hugging face hub
!pip install huggingface_hub -q

### **Use Python code to import the 'hf_hub_download' function from the 'huggingface_hub' library and also imports the 'Llama' class from the 'llama_cpp' library.**


In [3]:
!sudo find /usr/ -name 'libcuda.so.*'

/usr/local/cuda-12.2/compat/libcuda.so.1
/usr/local/cuda-12.2/compat/libcuda.so.535.129.03
/usr/lib64-nvidia/libcuda.so.535.104.05
/usr/lib64-nvidia/libcuda.so.1


In [4]:
!sudo apt install libcuda1 #  needed

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package libcuda1 is a virtual package provided by:
  libnvidia-compute-535 535.161.08-0ubuntu1 (= 535.161.08-0ubuntu1)
  libnvidia-compute-545 545.23.08-0ubuntu1 (= 545.23.08-0ubuntu1)
  libnvidia-compute-525 525.147.05-0ubuntu1 (= 525.147.05-0ubuntu1)
  libnvidia-compute-515 515.105.01-0ubuntu1 (= 515.105.01-0ubuntu1)
  libnvidia-compute-520 520.61.05-0ubuntu1 (= 520.61.05-0ubuntu1)
  libnvidia-compute-470-server 470.239.06-0ubuntu0.22.04.1 (= 470.239.06-0ubuntu0.22.04.1)
  libnvidia-compute-470 470.239.06-0ubuntu0.22.04.1 (= 470.239.06-0ubuntu0.22.04.1)
  libnvidia-compute-450-server 450.248.02-0ubuntu0.22.04.1 (= 450.248.02-0ubuntu0.22.04.1)
  libnvidia-compute-418-server 418.226.00-0ubuntu5~0.22.04.1 (= 418.226.00-0ubuntu5~0.22.04.1)
  libnvidia-compute-390 390.157-0ubuntu0.22.04.2 (= 390.157-0ubuntu0.22.04.2)
You should explicitly select one to install.

E: Package 'libcuda1' has no in

In [5]:
# We will use the 'hf_hub_download' function from the 'huggingface_hub' library

from huggingface_hub import hf_hub_download


In [6]:
!sudo apt-get -y install cuda-12-0  # not used - needs reboot

!nvcc --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cpp-12 cuda-cccl-12-0 cuda-command-line-tools-12-0 cuda-compiler-12-0
  cuda-cudart-12-0 cuda-cudart-dev-12-0 cuda-cuobjdump-12-0 cuda-cupti-12-0
  cuda-cupti-dev-12-0 cuda-cuxxfilt-12-0 cuda-demo-suite-12-0
  cuda-documentation-12-0 cuda-driver-dev-12-0 cuda-drivers cuda-drivers-550
  cuda-gdb-12-0 cuda-libraries-12-0 cuda-libraries-dev-12-0 cuda-nsight-12-0
  cuda-nsight-compute-12-0 cuda-nsight-systems-12-0 cuda-nvcc-12-0
  cuda-nvdisasm-12-0 cuda-nvml-dev-12-0 cuda-nvprof-12-0 cuda-nvprune-12-0
  cuda-nvrtc-12-0 cuda-nvrtc-dev-12-0 cuda-nvtx-12-0 cuda-nvvp-12-0
  cuda-opencl-12-0 cuda-opencl-dev-12-0 cuda-profiler-api-12-0
  cuda-runtime-12-0 cuda-sanitizer-12-0 cuda-toolkit-12-0
  cuda-toolkit-12-0-config-common cuda-tools-12-0 cuda-visual-tools-12-0
  dctrl-tools default-jre default-jre-headless dkms fakeroot fonts-dejavu-core
  f

In [7]:
# We will use the 'Llama' class from the 'llama_cpp' library

from llama_cpp import Llama

In [8]:
# Define the model name or path as a string (You can find this info from hugging face website)

model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"

# Define the model basename as a string, indicating it's in the gguf format

model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [9]:
# Download the model from the Hugging Face Hub using the 'hf_hub_download' function
# by specifying the 'repo_id' and 'filename'
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


llama-2-13b-chat.Q5_K_M.gguf:   0%|          | 0.00/9.23G [00:00<?, ?B/s]

In [10]:
# Create an instance of the 'Llama' class with specified parameters
# remove the blank spaces and complete the code

lcpp_llm = Llama(
        model_path=model_path,
        n_threads=2,  # CPU cores
        n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
        n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
        n_ctx=4096,  # Context window
    )

llama_model_loader: loaded meta data with 19 key-value pairs and 363 tensors from /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 5120
llama_model_loader: - kv   4:                          llama.block_count u32              = 40
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 13824
llama_model_loader: - kv   6:                 llama.rope.dimension_

In [34]:
import re

In [151]:
def generate_llama_response(message, prompt, problem):
    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=f"{message}{prompt}{problem}",
        max_tokens=256,
        temperature=0.5,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    rawResponse = response['choices'][0]['text']
    try:
      extracted = re.search('{.*}', rawResponse).group(0)
      return extracted
    except AttributeError:
      return ''

### **Load the input dataset into a dataframe**

In [13]:
# Import the pandas library and alias it as 'pd'
import pandas as pd


In [14]:

# Read a CSV file into a DataFrame and store it in the 'data' variable
# import the reviews into a dataframe

# Mount Google drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

# Install the openpyxl library
!pip install openpyxl

# Read the XLS file into a DataFrame
data = pd.read_excel('/content/drive/MyDrive/ColabFiles/ARTI-330/Support_ticket_text_data.xls')

Mounted at /content/drive


In [53]:
def remNewLine(text):
  return re.sub("\n", "", text)

In [141]:
prompt = """The guide is the numbered list. The numbered list shows the keys for each key value pair in the JSON object. Fill in the values for all 5 keys based on the support ticket, then return the JSON.
    (1) IC: Based on support ticket choose either technical, hardware or data.
    (2) TAG: Provide one to three relevant tags or keywords related to the problem.
    (3) PRIORITY: Assign a priority level (critical, high, medium, low) based on the severity and impact of the issue.
    (4) ERT: Provide an estimated time required to resolve the problem, expressed in hours or minutes.
    (5) REPLY: Draft a concise (25 word), response to the client, addressing the problem and providing relevant recommendations or troubleshooting steps.
    """

system_message = """Respond to the Support Ticket in JSON format. Reference the guide when doing so. This is going through an algorithm on the backend so do not include anything but the JSON object (!this means NO text outside the {} brackets)."""

problem = lambda ticket_text : f"Support Ticket To Base JSON return on: <|{ticket_text}|>"

In [58]:
import json

In [165]:
# Function to parse JSON data and extract key-value pairs
def extract_json_data(ticket):
    while True:
      try:
        resp = (json.loads(generate_llama_response(system_message, remNewLine(prompt), problem(ticket))))
        return resp
      except json.JSONDecodeError:
        continue
      except Exception:
        return None

In [152]:
test1 = generate_llama_response(system_message, remNewLine(prompt), problem(data['support_ticket_text'][10]))

Llama.generate: prefix-match hit

llama_print_timings:        load time =   10473.46 ms
llama_print_timings:      sample time =      59.33 ms /   101 runs   (    0.59 ms per token,  1702.46 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    1501.92 ms /   101 runs   (   14.87 ms per token,    67.25 tokens per second)
llama_print_timings:       total time =    1831.47 ms /   102 tokens


In [161]:
print(test1)

{ "IC": "data", "TAG": ["password reset", "online banking"], "PRIORITY": "high", "ERT": "2 hours", "REPLY": "We apologize for any inconvenience. Please click this link to reset your password and regain access to your online banking account: [insert link]. If you have any further issues, please reach out." }


In [166]:
test2 = extract_json_data(data['support_ticket_text'][10])

Llama.generate: prefix-match hit

llama_print_timings:        load time =   10473.46 ms
llama_print_timings:      sample time =       0.65 ms /     1 runs   (    0.65 ms per token,  1547.99 tokens per second)
llama_print_timings: prompt eval time =     107.19 ms /   176 tokens (    0.61 ms per token,  1642.01 tokens per second)
llama_print_timings:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_print_timings:       total time =     112.25 ms /   177 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =   10473.46 ms
llama_print_timings:      sample time =       0.68 ms /     1 runs   (    0.68 ms per token,  1466.28 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =      17.39 ms /     1 runs   (   17.39 ms per token,    57.49 tokens per second)
llama_print_timings:       to

In [167]:
print(test2)

{'IC': 'data', 'TAG': ['online banking', 'password reset'], 'PRIORITY': 'high', 'ERT': '1 hour', 'REPLY': 'We apologize for any inconvenience caused. Our team is ready to assist you with password reset and regaining access to your online banking account. Please provide the necessary information to verify your identity, and we will guide you through the process.'}


Now that it's done being tested, we can actually use it.

In [168]:
modelOutput = data['support_ticket_text'].apply(lambda ticket: extract_json_data(ticket))

Llama.generate: prefix-match hit

llama_print_timings:        load time =   10473.46 ms
llama_print_timings:      sample time =      63.13 ms /   110 runs   (    0.57 ms per token,  1742.55 tokens per second)
llama_print_timings: prompt eval time =     101.77 ms /    13 tokens (    7.83 ms per token,   127.75 tokens per second)
llama_print_timings:        eval time =    1597.57 ms /   109 runs   (   14.66 ms per token,    68.23 tokens per second)
llama_print_timings:       total time =    2054.80 ms /   122 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =   10473.46 ms
llama_print_timings:      sample time =     149.05 ms /   256 runs   (    0.58 ms per token,  1717.51 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    3767.47 ms /   256 runs   (   14.72 ms per token,    67.95 tokens per second)
llama_print_timings:       to

In [169]:
modelOutput

0     {'IC': 'data', 'TAG': ['network'], 'PRIORITY':...
1     {'IC': 'hardware', 'TAG': ['data recovery', 'l...
2     {'IC': 'technical', 'TAG': ['display settings'...
3     {'IC': 'technical', 'TAG': ['virus', 'malware'...
4     {'IC': 'hardware', 'TAG': ['printer issues'], ...
5     {'IC': 'data', 'TAG': ['internet connection sl...
6     {'IC': 'hardware', 'TAG': ['laptop start-up is...
7     {'IC': 'data', 'TAG': ['Data Recovery', 'Docum...
8     {'IC': 'data', 'TAG': ['Wi-Fi', 'signal streng...
9     {'IC': 'hardware', 'TAG': ['battery drain', 's...
10    {'IC': 'data', 'TAG': ['password reset', 'onli...
11    {'IC': 'data', 'TAG': ['performance', 'optimiz...
12    {'IC': 'hardware', 'TAG': ['blue screen', 'cra...
13    {'IC': 'data', 'TAG': ['external hard drive', ...
14    {'IC': 'hardware', 'TAG': ['gaming laptop', 'g...
15    {'IC': 'data', 'TAG': ['data recovery', 'USB d...
16    {'IC': 'hardware', 'TAG': ['monitor', 'black_s...
17    {'IC': 'hardware', 'TAG': ['water damage',

### Adding columns, from llm response

Category

In [171]:
data['category'] = modelOutput.apply(lambda row: row.get("IC"))

Tags

In [172]:
data['tag'] = modelOutput.apply(lambda row: row.get("TAG"))

Priority

In [173]:
data['priority'] = modelOutput.apply(lambda row: row.get("PRIORITY"))

Estimated resolution time

In [174]:
data['estimated_resolution_time'] = modelOutput.apply(lambda row: row.get("ERT"))

Generated reply

In [175]:
data['reply'] = modelOutput.apply(lambda row: row.get('REPLY'))

In [176]:
data.head()

,support_tick_id,support_ticket_text,category,tag,priority,estimated_resolution_time,reply
0,ST2024-001,How do you find the mac address on my computer?,data,[network],medium,2 hours,"To find your Mac Address, click on the Apple m..."
1,ST2024-002,I dropped my laptop in the parking lot and it ...,hardware,"[data recovery, laptop repair]",high,3-4 hours,Sorry to hear that you dropped your laptop and...
2,ST2024-003,The screen resolution on my computer monitor d...,technical,[display settings],low,30 minutes,Adjust your display settings to a higher resol...
3,ST2024-004,How do you get a virus off of my computer? I c...,technical,"[virus, malware]",high,2 hours,"We understand your concern, and we're here to ..."
4,ST2024-005,I can't get my computer to print to my printer...,hardware,[printer issues],medium,2 hours,Please check if your printer's USB cable is pr...
